In [6]:
# %%
import logging
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Project root = thư mục cha của notebooks/
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config").exists():
    raise RuntimeError("Could not locate project root from current working directory")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Python: {sys.version.split()[0]}")

# Set DEBUG để thấy mọi log từ src/ modules
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
# Tắt noise từ thư viện bên ngoài
for _lib in ["matplotlib", "PIL", "lightgbm", "mlflow", "urllib3"]:
    logging.getLogger(_lib).setLevel(logging.WARNING)

warnings.filterwarnings("ignore", category=UserWarning)
print("✅ Logging ready (DEBUG)")

✅ Project root: C:\Users\ADMIN\Documents\project_churn_prediction
✅ Python: 3.11.9
✅ Logging ready (DEBUG)


In [7]:
# %%
from src.utils.config import load_config

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(CONFIG_PATH)

print(f"📄 Config: {CONFIG_PATH}")
print(f"   random_state    : {config['project']['random_state']}")
print(f"   target_col      : {config['project']['target_col']}")
print(f"   candidates      : {config['modeling']['candidate_models']}")
print(f"   champion_metric : {config['modeling']['champion_metric']}")
print(f"   threshold       : {config['modeling']['decision_threshold']}")

02:05:31 | INFO     | src.utils.config | Config loaded from: C:\Users\ADMIN\Documents\project_churn_prediction\config\config.yaml
📄 Config: C:\Users\ADMIN\Documents\project_churn_prediction\config\config.yaml
   random_state    : 42
   target_col      : is_churn
   candidates      : ['logistic_regression', 'random_forest', 'lightgbm']
   champion_metric : auc_pr
   threshold       : 0.5


In [8]:
# %%
from sklearn.metrics import (
    average_precision_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)

THRESHOLD = float(config["modeling"]["decision_threshold"])

def compute_metrics(y_true, y_score, threshold=THRESHOLD) -> dict:
    y_pred  = (y_score >= threshold).astype(int)
    n_top   = max(1, int(len(y_true) * 0.10))
    top_idx = np.argsort(y_score)[::-1][:n_top]
    lift_10 = float(np.array(y_true)[top_idx].mean()) / float(y_true.mean())
    return {
        "AUC-ROC":   round(float(roc_auc_score(y_true, y_score)), 4),
        "AUC-PR":    round(float(average_precision_score(y_true, y_score)), 4),
        "F1":        round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        "Precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "Recall":    round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "Lift@10%":  round(lift_10, 2),
    }

def print_metrics(name: str, train_m: dict, val_m: dict):
    print(f"\n{'─'*58}")
    print(f"  📈  {name}")
    print(f"{'─'*58}")
    print(f"  {'Metric':<14} {'Train':>10} {'Val':>10}  {'Gap':>8}")
    print(f"  {'─'*14} {'─'*10} {'─'*10}  {'─'*8}")
    for k in train_m:
        gap  = train_m[k] - val_m[k]
        flag = " ⚠️" if abs(gap) > 0.05 and k in ("AUC-ROC","AUC-PR") else ""
        print(f"  {k:<14} {train_m[k]:>10.4f} {val_m[k]:>10.4f}"
              f"  {gap:>+8.4f}{flag}")
    print(f"{'─'*58}\n")

print(f"✅ Metric helpers ready  |  threshold = {THRESHOLD}")

✅ Metric helpers ready  |  threshold = 0.5


In [9]:
# %%
# Load persisted splits and preprocessor (produced by src/features/run_preprocessing.py)
from pathlib import Path
from src.features.preprocess import load_splits, load_preprocessor

processed_dir = PROJECT_ROOT / "data" / "processed"
preprocessor_path = PROJECT_ROOT / "models" / "preprocessor.pkl"

try:
    splits = load_splits(processed_dir)
    X_train = splits.X_train
    X_val = splits.X_val
    X_test = splits.X_test
    y_train = splits.y_train
    y_val = splits.y_val
    y_test = splits.y_test
    preprocessor = load_preprocessor(preprocessor_path)
    print(f"Loaded splits from {processed_dir} and preprocessor from {preprocessor_path}")
except FileNotFoundError as err:
    raise RuntimeError(f"Missing preprocessing artifacts: {err}\nRun src/features/run_preprocessing.py first")


02:05:31 | INFO     | src.features.preprocess | Loaded X_train from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\X_train.parquet  (shape=(695051, 54))
02:05:31 | INFO     | src.features.preprocess | Loaded X_val from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\X_val.parquet  (shape=(148940, 54))
02:05:31 | INFO     | src.features.preprocess | Loaded X_test from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\X_test.parquet  (shape=(148940, 54))
02:05:31 | INFO     | src.features.preprocess | Loaded y_train from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\y_train.parquet  (shape=(695051, 1))
02:05:31 | INFO     | src.features.preprocess | Loaded y_val from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\y_val.parquet  (shape=(148940, 1))
02:05:31 | INFO     | src.features.preprocess | Loaded y_test from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\y_test.parquet  (shape=(1

In [ ]:
# %%
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

print("🚀 Training Logistic Regression (baseline)...")

lr_params = {
    "solver":       config["logistic_regression"]["solver"],
    "max_iter":     int(config["logistic_regression"]["max_iter"]),
    "class_weight": config["logistic_regression"]["class_weight"],
    "random_state": int(config["project"]["random_state"]),
    "n_jobs":       -1,
    "verbose":      1,   # ← in convergence info từ solver
}
print(f"   params: {lr_params}\n")

# If any NaNs remain in the loaded, already-transformed splits, impute them
if X_train.isnull().values.any() or X_val.isnull().values.any() or X_test.isnull().values.any():
    print("🩹 Detected NaNs in splits — imputing with median (fit on train)")
    imputer = SimpleImputer(strategy="median")
    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    X_val = pd.DataFrame(
        imputer.transform(X_val),
        columns=X_val.columns,
        index=X_val.index,
    )
    X_test = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )
    print("🩹 Imputation complete")

t0 = time.perf_counter()
lr_model = LogisticRegression(**lr_params)
lr_model.fit(X_train, y_train)
lr_time = time.perf_counter() - t0

print(f"\n⏱  Done in {lr_time:.1f}s  "
      f"| n_iter={lr_model.n_iter_[0]}/{lr_params['max_iter']}")

lr_train_m = compute_metrics(y_train, lr_model.predict_proba(X_train)[:, 1])
lr_val_m   = compute_metrics(y_val,   lr_model.predict_proba(X_val)[:, 1])
print_metrics("Logistic Regression (Baseline)", lr_train_m, lr_val_m)


🚀 Training Logistic Regression (baseline)...
   params: {'solver': 'lbfgs', 'max_iter': 1000, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1, 'verbose': 1}



c:\Users\ADMIN\Documents\project_churn_prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values